In [14]:
import d3rlpy
from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer
from d3rlpy.algos.transformer.decision_transformer import DTConstantRTGforFQE,DecisionTransformer
from d3rlpy.ope.fqe import FQE,FQEConfig
import time

on_server = False  # Set to True if running on the server
prefix = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/" if on_server else "/home/julian/programming/cloned_repos/repos_for_master_thesis/"
cql_model = prefix + "d3rlpy/experiments/exp02_qdt/d3rlpy_logs/CQL_Hopper-v4_1_20250710203053/model_400000.d3"
qdt_model = prefix + "d3rlpy/experiments/exp02_qdt/d3rlpy_logs/QDT_hopper-medium-expert-v2_1_20250710151721/model_epoch_14.d3"

start_time = time.time()


dt_model = prefix + "d3rlpy/experiments/exp01_original_dt/slurm_files/d3rlpy_logs/gpu_array/DT_hopper-medium-expert-v2_1_20250710202727/model_epoch_12.d3"

device = "cuda:0" if on_server else "cpu"
dt_algo = d3rlpy.load_learnable(dt_model,device=device)
cql_algo = d3rlpy.load_learnable(cql_model,device=device)
qdt_algo = d3rlpy.load_learnable(qdt_model,device=device)

In [7]:
from d3rlpy.metrics import evaluate_transformer_with_environment
import gymnasium as gym
def get_env_and_target_return(dataset):    
    if "halfcheetah" in dataset:
        return "HalfCheetah-v4", 6000
    elif "hopper" in dataset:
        return "Hopper-v4", 3800
    elif "walker" in dataset:
        return "Walker2d-v4", 5000
dataset= "hopper-medium-expert-v2"
env_name, target_return = get_env_and_target_return(dataset)
env = gym.make(env_name)

/home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gymnasium/envs/registration.py:517: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


In [16]:
#evaluate the Decision Transformer with the environment
dt_eval_dict = evaluate_transformer_with_environment(
algo=dt_algo.as_stateful_wrapper(
    target_return=target_return,
),
env=env,
n_trials=10)

In [18]:
#evaluate the QDT with the environment
qdt_eval_dict = evaluate_transformer_with_environment(
algo=qdt_algo.as_stateful_wrapper(
    target_return=target_return,
),
env=env,
n_trials=10)

In [10]:
#Evaluate the CQL model with the environment
from d3rlpy.dataset import ReplayBuffer
env_eval = d3rlpy.metrics.EnvironmentEvaluator(env, n_trials=10)
mean_return = env_eval(cql_algo,ReplayBuffer)

In [19]:
mean_return_dt = dt_eval_dict["episode_mean_reward"]
mean_return_cql = mean_return
mean_return_qdt = qdt_eval_dict["episode_mean_reward"]
print(f"Mean return DT: {mean_return_dt}, Mean return CQL: {mean_return_cql}, Mean return QDT: {mean_return_qdt}")

Mean return DT: 3207.8253878827563, Mean return CQL: 2941.6932094161266, Mean return QDT: 1978.8441019090274
